<a href="https://colab.research.google.com/github/KunalGaurav90/Sap_project-O2C-/blob/main/Sap_project(O2C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
#   KIIT SAP PROJECT — ORDER-TO-CASH (O2C) SIMULATION
#   Topic ii: Complete Sales Cycle (SAP SD)
# ============================================================

# ── INSTALL & IMPORTS ────────────────────────────────────────

import datetime
import random
from tabulate import tabulate

print("=" * 62)
print("   KIIT SAP PROJECT — ORDER-TO-CASH (O2C) SIMULATION")
print("   SAP SD: Complete Sales Cycle")
print("=" * 62)

   KIIT SAP PROJECT — ORDER-TO-CASH (O2C) SIMULATION
   SAP SD: Complete Sales Cycle


In [3]:

# ============================================================
# SECTION 1: MASTER DATA SETUP
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 1: MASTER DATA SETUP")
print("═" * 62)

# ── Company Master ───────────────────────────────────────────
company = {
    "company_code": "TN01",
    "company_name": "TechNova Solutions Pvt. Ltd Solution.",
    "sales_org": "TN10",
    "dist_channel": "10",
    "division": "00",
    "plant": "TN01",
    "storage_location": "0008",
    "shipping_point": "TN01",
    "currency": "INR",
}

print("\n[COMPANY MASTER DATA]")
for k, v in company.items():
    print(f"  {k:<25} : {v}")

# ── Customer Master ───────────────────────────────────────────
customers = {
    "CUST-1001": {
        "name": "GlobalTech Enterprises",
        "group": "Domestic",
        "payment_terms": "Net 30 days",
        "credit_limit": 2_000_000,
        "open_ar": 0,
        "incoterms": "EXW",
        "currency": "INR",
    }
}

print("\n[CUSTOMER MASTER DATA]")
c = customers["CUST-1001"]
for k, v in c.items():
    print(f"  {k:<25} : {v}")

# ── Material Master ───────────────────────────────────────────
materials = {
    "LAPT-1001": {
        "description": "Laptop Pro X15",
        "type": "FERT",
        "uom": "EA",
        "base_price": 75_000,
        "tax_rate": 0.18,
        "stock": 50,
    }
}

print("\n[MATERIAL MASTER DATA]")
m = materials["LAPT-1001"]
for k, v in m.items():
    print(f"  {k:<25} : {v}")

# ── Document Counter (simulates SAP number ranges) ────────────
doc_counter = {"INQ": 10000, "QT": 20000, "SO": 30000,
               "DEL": 40000, "MAT": 50000, "BILL": 60000, "FI": 70000}

def next_doc(doc_type):
    doc_counter[doc_type] += 1
    return f"{doc_type}-{doc_counter[doc_type]}"

# ── Document Store (simulates SAP database) ───────────────────
documents = {}


══════════════════════════════════════════════════════════════
  SECTION 1: MASTER DATA SETUP
══════════════════════════════════════════════════════════════

[COMPANY MASTER DATA]
  company_code              : TN01
  company_name              : TechNova Solutions Pvt. Ltd Solution.
  sales_org                 : TN10
  dist_channel              : 10
  division                  : 00
  plant                     : TN01
  storage_location          : 0008
  shipping_point            : TN01
  currency                  : INR

[CUSTOMER MASTER DATA]
  name                      : GlobalTech Enterprises
  group                     : Domestic
  payment_terms             : Net 30 days
  credit_limit              : 2000000
  open_ar                   : 0
  incoterms                 : EXW
  currency                  : INR

[MATERIAL MASTER DATA]
  description               : Laptop Pro X15
  type                      : FERT
  uom                       : EA
  base_price                : 75000
  tax_r

In [4]:

# ============================================================
# SECTION 2: PRE-SALES — INQUIRY (VA11)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 2: PRE-SALES — INQUIRY  [T-Code: VA11]")
print("═" * 62)

def create_inquiry(customer_id, material_id, quantity, sales_area):
    """
    Simulates VA11 — Create Inquiry in SAP SD.
    An inquiry is a customer's request for information about
    product pricing and availability (non-binding).
    """
    if customer_id not in customers:
        raise ValueError(f"Customer {customer_id} not found.")
    if material_id not in materials:
        raise ValueError(f"Material {material_id} not found.")

    mat = materials[material_id]
    doc_no = next_doc("INQ")
    inquiry = {
        "doc_no": doc_no,
        "doc_type": "IN",
        "status": "Open",
        "customer": customer_id,
        "customer_name": customers[customer_id]["name"],
        "sales_org": sales_area["sales_org"],
        "dist_channel": sales_area["dist_channel"],
        "division": sales_area["division"],
        "date": datetime.date.today().isoformat(),
        "valid_from": datetime.date.today().isoformat(),
        "valid_to": (datetime.date.today() + datetime.timedelta(days=30)).isoformat(),
        "items": [
            {
                "item_no": "10",
                "material": material_id,
                "description": mat["description"],
                "quantity": quantity,
                "uom": mat["uom"],
                "unit_price": mat["base_price"],
                "net_value": quantity * mat["base_price"],
            }
        ],
    }
    documents[doc_no] = inquiry
    print(f"\n  [VA11] Inquiry created successfully.")
    print(f"  Document Number : {doc_no}")
    print(f"  Customer        : {customers[customer_id]['name']}")
    print(f"  Material        : {mat['description']}")
    print(f"  Quantity        : {quantity} {mat['uom']}")
    print(f"  Estimated Value : INR {quantity * mat['base_price']:,.0f}")
    return doc_no

inquiry_no = create_inquiry(
    customer_id="CUST-1001",
    material_id="LAPT-1001",
    quantity=10,
    sales_area={"sales_org": "TN10", "dist_channel": "10", "division": "00"}
)


══════════════════════════════════════════════════════════════
  SECTION 2: PRE-SALES — INQUIRY  [T-Code: VA11]
══════════════════════════════════════════════════════════════

  [VA11] Inquiry created successfully.
  Document Number : INQ-10001
  Customer        : GlobalTech Enterprises
  Material        : Laptop Pro X15
  Quantity        : 10 EA
  Estimated Value : INR 750,000


In [5]:
# ============================================================
# SECTION 3: PRE-SALES — QUOTATION (VA21)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 3: PRE-SALES — QUOTATION  [T-Code: VA21]")
print("═" * 62)

def create_quotation(inquiry_no, discount_pct=0):
    """
    Simulates VA21 — Create Quotation with reference to Inquiry.
    A quotation is a binding offer specifying price and delivery terms.
    Discount can optionally be applied.
    """
    if inquiry_no not in documents:
        raise ValueError(f"Inquiry {inquiry_no} not found.")

    inq = documents[inquiry_no]
    doc_no = next_doc("QT")
    items = []
    for item in inq["items"]:
        discount = item["net_value"] * (discount_pct / 100)
        net_value = item["net_value"] - discount
        items.append({**item,
                      "discount_pct": discount_pct,
                      "discount_value": discount,
                      "net_value": net_value})

    quotation = {
        "doc_no": doc_no,
        "doc_type": "QT",
        "status": "Open",
        "ref_inquiry": inquiry_no,
        "customer": inq["customer"],
        "customer_name": inq["customer_name"],
        "sales_org": inq["sales_org"],
        "date": datetime.date.today().isoformat(),
        "valid_from": datetime.date.today().isoformat(),
        "valid_to": (datetime.date.today() + datetime.timedelta(days=30)).isoformat(),
        "payment_terms": "NT30",
        "incoterms": "EXW",
        "items": items,
        "total_net_value": sum(i["net_value"] for i in items),
    }
    documents[doc_no] = quotation
    inq["status"] = "Completed"

    print(f"\n  [VA21] Quotation created with reference to Inquiry {inquiry_no}")
    print(f"  Document Number   : {doc_no}")
    print(f"  Valid Until       : {quotation['valid_to']}")
    print(f"  Payment Terms     : {quotation['payment_terms']}")
    print(f"  Discount Applied  : {discount_pct}%")
    print(f"  Total Net Value   : INR {quotation['total_net_value']:,.0f}")
    return doc_no

quotation_no = create_quotation(inquiry_no, discount_pct=5)  # 5% discount


══════════════════════════════════════════════════════════════
  SECTION 3: PRE-SALES — QUOTATION  [T-Code: VA21]
══════════════════════════════════════════════════════════════

  [VA21] Quotation created with reference to Inquiry INQ-10001
  Document Number   : QT-20001
  Valid Until       : 2026-05-20
  Payment Terms     : NT30
  Discount Applied  : 5%
  Total Net Value   : INR 712,500


In [6]:
# ============================================================
# SECTION 4: SALES ORDER (VA01)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 4: SALES ORDER  [T-Code: VA01]")
print("═" * 62)

def create_sales_order(quotation_no, customer_po="PO-GLOBTECH-001"):
    """
    Simulates VA01 — Create Sales Order with reference to Quotation.
    The Sales Order is the central document in the O2C cycle.
    It triggers credit check, availability check, and delivery.
    """
    if quotation_no not in documents:
        raise ValueError(f"Quotation {quotation_no} not found.")

    qt = documents[quotation_no]
    cust = customers[qt["customer"]]
    doc_no = next_doc("SO")

    # Credit Check Simulation
    total_exposure = cust["open_ar"] + qt["total_net_value"]
    credit_ok = total_exposure <= cust["credit_limit"]
    print(f"\n  [CREDIT CHECK]")
    print(f"  Credit Limit    : INR {cust['credit_limit']:,.0f}")
    print(f"  Current A/R     : INR {cust['open_ar']:,.0f}")
    print(f"  New Exposure    : INR {qt['total_net_value']:,.0f}")
    print(f"  Total Exposure  : INR {total_exposure:,.0f}")
    print(f"  Credit Result   : {'✓ APPROVED' if credit_ok else '✗ BLOCKED'}")

    if not credit_ok:
        raise Exception("Sales Order BLOCKED: Credit limit exceeded.")

    # Availability Check Simulation
    print(f"\n  [AVAILABILITY CHECK]")
    for item in qt["items"]:
        mat = materials[item["material"]]
        available = mat["stock"] >= item["quantity"]
        print(f"  Material : {mat['description']}")
        print(f"  Required : {item['quantity']} {mat['uom']}")
        print(f"  In Stock : {mat['stock']} {mat['uom']}")
        print(f"  Status   : {'✓ CONFIRMED' if available else '✗ PARTIAL/BACKORDER'}")

    delivery_date = datetime.date.today() + datetime.timedelta(days=7)
    items = []
    for item in qt["items"]:
        mat = materials[item["material"]]
        tax = item["net_value"] * mat["tax_rate"]
        items.append({**item,
                      "tax_pct": mat["tax_rate"] * 100,
                      "tax_value": tax,
                      "gross_value": item["net_value"] + tax,
                      "confirmed_qty": item["quantity"],
                      "delivery_date": delivery_date.isoformat(),
                      "plant": company["plant"],
                      "storage_loc": company["storage_location"],
                      "item_category": "TAN"})

    sales_order = {
        "doc_no": doc_no,
        "doc_type": "OR",
        "status": "Open",
        "ref_quotation": quotation_no,
        "customer": qt["customer"],
        "customer_name": qt["customer_name"],
        "customer_po": customer_po,
        "sales_org": qt["sales_org"],
        "date": datetime.date.today().isoformat(),
        "req_delivery_date": delivery_date.isoformat(),
        "payment_terms": qt["payment_terms"],
        "incoterms": qt["incoterms"],
        "items": items,
        "total_net_value": qt["total_net_value"],
        "total_tax": sum(i["tax_value"] for i in items),
        "total_gross": sum(i["gross_value"] for i in items),
    }
    documents[doc_no] = sales_order
    documents[quotation_no]["status"] = "Completed"

    print(f"\n  [VA01] Sales Order created successfully.")
    print(f"  Document Number       : {doc_no}")
    print(f"  Customer PO           : {customer_po}")
    print(f"  Requested Delivery    : {delivery_date.isoformat()}")
    print(f"  Net Value             : INR {sales_order['total_net_value']:,.0f}")
    print(f"  Tax (GST 18%)         : INR {sales_order['total_tax']:,.0f}")
    print(f"  Total Gross Value     : INR {sales_order['total_gross']:,.0f}")
    return doc_no

sales_order_no = create_sales_order(quotation_no)


══════════════════════════════════════════════════════════════
  SECTION 4: SALES ORDER  [T-Code: VA01]
══════════════════════════════════════════════════════════════

  [CREDIT CHECK]
  Credit Limit    : INR 2,000,000
  Current A/R     : INR 0
  New Exposure    : INR 712,500
  Total Exposure  : INR 712,500
  Credit Result   : ✓ APPROVED

  [AVAILABILITY CHECK]
  Material : Laptop Pro X15
  Required : 10 EA
  In Stock : 50 EA
  Status   : ✓ CONFIRMED

  [VA01] Sales Order created successfully.
  Document Number       : SO-30001
  Customer PO           : PO-GLOBTECH-001
  Requested Delivery    : 2026-04-27
  Net Value             : INR 712,500
  Tax (GST 18%)         : INR 128,250
  Total Gross Value     : INR 840,750


In [7]:
# ============================================================
# SECTION 5: DELIVERY CREATION (VL01N)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 5: DELIVERY CREATION  [T-Code: VL01N]")
print("═" * 62)

def create_delivery(so_no):
    """
    Simulates VL01N — Create Outbound Delivery.
    Delivery document picks, packs, and ships goods to customer.
    """
    if so_no not in documents:
        raise ValueError(f"Sales Order {so_no} not found.")

    so = documents[so_no]
    if so["status"] != "Open":
        raise Exception(f"Sales Order {so_no} is not open for delivery.")

    doc_no = next_doc("DEL")
    delivery = {
        "doc_no": doc_no,
        "doc_type": "LF",
        "status": "Open",
        "ref_so": so_no,
        "customer": so["customer"],
        "customer_name": so["customer_name"],
        "shipping_point": company["shipping_point"],
        "delivery_date": so["req_delivery_date"],
        "date_created": datetime.date.today().isoformat(),
        "items": [],
        "pgi_posted": False,
    }
    for item in so["items"]:
        delivery["items"].append({
            "item_no": item["item_no"],
            "material": item["material"],
            "description": item["description"],
            "delivery_qty": item["confirmed_qty"],
            "uom": item["uom"],
            "plant": item["plant"],
            "storage_loc": item["storage_loc"],
            "pick_status": "To Pick",
            "pack_status": "To Pack",
        })

    documents[doc_no] = delivery
    print(f"\n  [VL01N] Outbound Delivery created.")
    print(f"  Document Number   : {doc_no}")
    print(f"  Ship-to Party     : {so['customer_name']}")
    print(f"  Shipping Point    : {company['shipping_point']}")
    print(f"  Delivery Date     : {so['req_delivery_date']}")
    print(f"  Items to Ship:")
    for item in delivery["items"]:
        print(f"    {item['material']} — {item['description']} x {item['delivery_qty']} {item['uom']}")
    return doc_no

delivery_no = create_delivery(sales_order_no)


══════════════════════════════════════════════════════════════
  SECTION 5: DELIVERY CREATION  [T-Code: VL01N]
══════════════════════════════════════════════════════════════

  [VL01N] Outbound Delivery created.
  Document Number   : DEL-40001
  Ship-to Party     : GlobalTech Enterprises
  Shipping Point    : TN01
  Delivery Date     : 2026-04-27
  Items to Ship:
    LAPT-1001 — Laptop Pro X15 x 10 EA


In [8]:
# ============================================================
# SECTION 6: POST GOODS ISSUE (VL02N)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 6: POST GOODS ISSUE (PGI)  [T-Code: VL02N]")
print("═" * 62)

def post_goods_issue(delivery_no):
    """
    Simulates VL02N — Post Goods Issue (PGI).
    PGI reduces inventory and triggers COGS accounting entry.
    This is the point of legal ownership transfer to the customer.
    """
    if delivery_no not in documents:
        raise ValueError(f"Delivery {delivery_no} not found.")

    deliv = documents[delivery_no]
    if deliv["pgi_posted"]:
        raise Exception("PGI already posted for this delivery.")

    mat_doc_no = next_doc("MAT")
    print(f"\n  [VL02N] Posting Goods Issue...")

    stock_movements = []
    for item in deliv["items"]:
        mat = materials[item["material"]]
        old_stock = mat["stock"]
        mat["stock"] -= item["delivery_qty"]
        stock_movements.append({
            "material": item["material"],
            "description": item["description"],
            "movement_type": "601",
            "qty": item["delivery_qty"],
            "from_stock": old_stock,
            "to_stock": mat["stock"],
            "cogs": item["delivery_qty"] * mat["base_price"],
        })
        item["pick_status"] = "Picked"
        item["pack_status"] = "Packed"

    deliv["pgi_posted"] = True
    deliv["pgi_date"] = datetime.date.today().isoformat()
    deliv["status"] = "Goods Issue Completed"
    deliv["material_doc"] = mat_doc_no

    documents[mat_doc_no] = {
        "doc_no": mat_doc_no,
        "type": "Material Document",
        "movement_type": "601",
        "movements": stock_movements,
    }

    print(f"  Material Document : {mat_doc_no}")
    print(f"  PGI Date          : {deliv['pgi_date']}")
    print(f"  Delivery Status   : {deliv['status']}")
    print(f"\n  [STOCK IMPACT]")
    headers = ["Material", "Description", "Mvt Type", "Qty", "Old Stock", "New Stock"]
    rows = [[m["material"], m["description"], m["movement_type"],
             m["qty"], m["from_stock"], m["to_stock"]] for m in stock_movements]
    print(tabulate(rows, headers=headers, tablefmt="grid"))
    print(f"\n  [FI ENTRY — COGS]")
    total_cogs = sum(m["cogs"] for m in stock_movements)
    fi_rows = [
        ["COGS / Cost of Sales", "Debit", f"INR {total_cogs:,.0f}"],
        ["Finished Goods Inventory", "Credit", f"INR {total_cogs:,.0f}"],
    ]
    print(tabulate(fi_rows, headers=["Account", "Dr/Cr", "Amount"], tablefmt="grid"))
    return mat_doc_no

mat_doc_no = post_goods_issue(delivery_no)


══════════════════════════════════════════════════════════════
  SECTION 6: POST GOODS ISSUE (PGI)  [T-Code: VL02N]
══════════════════════════════════════════════════════════════

  [VL02N] Posting Goods Issue...
  Material Document : MAT-50001
  PGI Date          : 2026-04-20
  Delivery Status   : Goods Issue Completed

  [STOCK IMPACT]
+------------+----------------+------------+-------+-------------+-------------+
| Material   | Description    |   Mvt Type |   Qty |   Old Stock |   New Stock |
+============+================+============+=======+=============+=============+
| LAPT-1001  | Laptop Pro X15 |        601 |    10 |          50 |          40 |
+------------+----------------+------------+-------+-------------+-------------+

  [FI ENTRY — COGS]
+--------------------------+---------+-------------+
| Account                  | Dr/Cr   | Amount      |
+==========================+=========+=============+
| COGS / Cost of Sales     | Debit   | INR 750,000 |
+--------------------

In [9]:

# ============================================================
# SECTION 7: BILLING (VF01)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 7: BILLING / INVOICE  [T-Code: VF01]")
print("═" * 62)

def create_billing(delivery_no):
    """
    Simulates VF01 — Create Billing Document (Customer Invoice).
    Billing is based on the delivery and posts an AR entry in FI.
    Billing Type: F2 (Standard Invoice).
    """
    if delivery_no not in documents:
        raise ValueError(f"Delivery {delivery_no} not found.")

    deliv = documents[delivery_no]
    if not deliv["pgi_posted"]:
        raise Exception("Cannot bill: PGI not yet posted.")

    so = documents[deliv["ref_so"]]
    doc_no = next_doc("BILL")
    fi_doc_no = next_doc("FI")

    items = []
    total_net = 0
    total_tax = 0
    for so_item, del_item in zip(so["items"], deliv["items"]):
        net_val = so_item["net_value"]
        tax_val = so_item["tax_value"]
        gross_val = so_item["gross_value"]
        total_net += net_val
        total_tax += tax_val
        items.append({
            "item_no": so_item["item_no"],
            "material": so_item["material"],
            "description": so_item["description"],
            "billing_qty": del_item["delivery_qty"],
            "uom": so_item["uom"],
            "unit_price": so_item["unit_price"],
            "net_value": net_val,
            "tax_pct": so_item["tax_pct"],
            "tax_value": tax_val,
            "gross_value": gross_val,
        })

    total_gross = total_net + total_tax
    due_date = (datetime.date.today() + datetime.timedelta(days=30)).isoformat()

    billing_doc = {
        "doc_no": doc_no,
        "doc_type": "F2",
        "status": "Posted",
        "ref_delivery": delivery_no,
        "ref_so": deliv["ref_so"],
        "customer": deliv["customer"],
        "customer_name": deliv["customer_name"],
        "billing_date": datetime.date.today().isoformat(),
        "due_date": due_date,
        "payment_terms": so["payment_terms"],
        "items": items,
        "total_net_value": total_net,
        "total_tax": total_tax,
        "total_gross": total_gross,
        "fi_doc": fi_doc_no,
        "cleared": False,
    }
    documents[doc_no] = billing_doc

    # Update open AR
    customers[deliv["customer"]]["open_ar"] += total_gross

    # FI Accounting Entry
    print(f"\n  [VF01] Billing Document created.")
    print(f"  Document Number   : {doc_no}")
    print(f"  Billing Type      : F2 (Standard Invoice)")
    print(f"  Customer          : {deliv['customer_name']}")
    print(f"  Net Value         : INR {total_net:,.0f}")
    print(f"  GST (18%)         : INR {total_tax:,.0f}")
    print(f"  Gross Value       : INR {total_gross:,.0f}")
    print(f"  Due Date          : {due_date}")
    print(f"\n  [FI ACCOUNTING ENTRY — AUTO GENERATED]")
    fi_rows = [
        ["Customer Receivable (AR)", "Debit",  f"INR {total_gross:,.0f}"],
        ["Sales Revenue Account",    "Credit", f"INR {total_net:,.0f}"],
        ["GST Output Tax (18%)",     "Credit", f"INR {total_tax:,.0f}"],
    ]
    print(tabulate(fi_rows, headers=["Account", "Dr/Cr", "Amount"], tablefmt="grid"))
    print(f"  FI Document No    : {fi_doc_no}")
    return doc_no

billing_no = create_billing(delivery_no)


══════════════════════════════════════════════════════════════
  SECTION 7: BILLING / INVOICE  [T-Code: VF01]
══════════════════════════════════════════════════════════════

  [VF01] Billing Document created.
  Document Number   : BILL-60001
  Billing Type      : F2 (Standard Invoice)
  Customer          : GlobalTech Enterprises
  Net Value         : INR 712,500
  GST (18%)         : INR 128,250
  Gross Value       : INR 840,750
  Due Date          : 2026-05-20

  [FI ACCOUNTING ENTRY — AUTO GENERATED]
+--------------------------+---------+-------------+
| Account                  | Dr/Cr   | Amount      |
+==========================+=========+=============+
| Customer Receivable (AR) | Debit   | INR 840,750 |
+--------------------------+---------+-------------+
| Sales Revenue Account    | Credit  | INR 712,500 |
+--------------------------+---------+-------------+
| GST Output Tax (18%)     | Credit  | INR 128,250 |
+--------------------------+---------+-------------+
  FI Document 

In [10]:
# ============================================================
# SECTION 8: PAYMENT POSTING (F-28)
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 8: PAYMENT POSTING  [T-Code: F-28]")
print("═" * 62)

def post_payment(billing_no, amount_received=None):
    """
    Simulates F-28 — Post Incoming Payment.
    Clears open AR item and posts cash/bank entry.
    """
    if billing_no not in documents:
        raise ValueError(f"Billing Document {billing_no} not found.")

    bill = documents[billing_no]
    if bill["cleared"]:
        raise Exception("Invoice already cleared.")

    amount = amount_received or bill["total_gross"]
    fi_pay_doc = next_doc("FI")
    diff = bill["total_gross"] - amount
    fully_cleared = abs(diff) < 0.01

    bill["cleared"] = fully_cleared
    bill["payment_doc"] = fi_pay_doc
    bill["payment_date"] = datetime.date.today().isoformat()
    bill["amount_paid"] = amount

    # Reduce open AR
    customers[bill["customer"]]["open_ar"] -= amount

    print(f"\n  [F-28] Incoming Payment Posted.")
    print(f"  Payment Document  : {fi_pay_doc}")
    print(f"  Invoice Amount    : INR {bill['total_gross']:,.0f}")
    print(f"  Amount Received   : INR {amount:,.0f}")
    print(f"  Difference        : INR {diff:,.0f}")
    print(f"  Clearing Status   : {'✓ FULLY CLEARED' if fully_cleared else '⚠ PARTIAL PAYMENT'}")
    print(f"\n  [FI PAYMENT ENTRY]")
    fi_rows = [
        ["Bank / Cash Account",        "Debit",  f"INR {amount:,.0f}"],
        ["Customer Receivable (AR)",   "Credit", f"INR {amount:,.0f}"],
    ]
    print(tabulate(fi_rows, headers=["Account", "Dr/Cr", "Amount"], tablefmt="grid"))
    print(f"  Remaining A/R : INR {customers[bill['customer']]['open_ar']:,.0f}")
    return fi_pay_doc

payment_doc = post_payment(billing_no)



══════════════════════════════════════════════════════════════
  SECTION 8: PAYMENT POSTING  [T-Code: F-28]
══════════════════════════════════════════════════════════════

  [F-28] Incoming Payment Posted.
  Payment Document  : FI-70002
  Invoice Amount    : INR 840,750
  Amount Received   : INR 840,750
  Difference        : INR 0
  Clearing Status   : ✓ FULLY CLEARED

  [FI PAYMENT ENTRY]
+--------------------------+---------+-------------+
| Account                  | Dr/Cr   | Amount      |
+==========================+=========+=============+
| Bank / Cash Account      | Debit   | INR 840,750 |
+--------------------------+---------+-------------+
| Customer Receivable (AR) | Credit  | INR 840,750 |
+--------------------------+---------+-------------+
  Remaining A/R : INR 0


In [11]:
# ============================================================
# SECTION 9: FULL O2C DOCUMENT CHAIN REPORT
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 9: O2C DOCUMENT FLOW SUMMARY")
print("═" * 62)

def print_document_chain():
    inq  = documents[inquiry_no]
    qt   = documents[quotation_no]
    so   = documents[sales_order_no]
    dl   = documents[delivery_no]
    bill = documents[billing_no]

    chain = [
        ["1", "Inquiry",          inq["doc_no"],          "VA11", inq["status"]],
        ["2", "Quotation",        qt["doc_no"],           "VA21", qt["status"]],
        ["3", "Sales Order",      so["doc_no"],           "VA01", so["status"]],
        ["4", "Outbound Delivery",dl["doc_no"],           "VL01N",dl["status"]],
        ["5", "Material Document",dl["material_doc"],     "VL02N","Posted"],
        ["6", "Billing Document", bill["doc_no"],         "VF01", "Posted"],
        ["7", "FI AR Entry",      bill["fi_doc"],         "FB03", "Posted"],
        ["8", "Payment Doc",      bill["payment_doc"],    "F-28", "Cleared"],
    ]
    print()
    print(tabulate(chain,
                   headers=["Step", "Document Type", "Doc Number", "T-Code", "Status"],
                   tablefmt="fancy_grid"))

print_document_chain()


══════════════════════════════════════════════════════════════
  SECTION 9: O2C DOCUMENT FLOW SUMMARY
══════════════════════════════════════════════════════════════

╒════════╤═══════════════════╤══════════════╤══════════╤═══════════════════════╕
│   Step │ Document Type     │ Doc Number   │ T-Code   │ Status                │
╞════════╪═══════════════════╪══════════════╪══════════╪═══════════════════════╡
│      1 │ Inquiry           │ INQ-10001    │ VA11     │ Completed             │
├────────┼───────────────────┼──────────────┼──────────┼───────────────────────┤
│      2 │ Quotation         │ QT-20001     │ VA21     │ Completed             │
├────────┼───────────────────┼──────────────┼──────────┼───────────────────────┤
│      3 │ Sales Order       │ SO-30001     │ VA01     │ Open                  │
├────────┼───────────────────┼──────────────┼──────────┼───────────────────────┤
│      4 │ Outbound Delivery │ DEL-40001    │ VL01N    │ Goods Issue Completed │
├────────┼─────────────

In [12]:

# ============================================================
# SECTION 10: FINANCIAL SUMMARY
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 10: FINANCIAL SUMMARY")
print("═" * 62)

so   = documents[sales_order_no]
bill = documents[billing_no]

summary = [
    ["Gross Revenue (before discount)", f"INR {sum(i['quantity']*i['unit_price'] for i in documents[inquiry_no]['items']):,.0f}"],
    ["Discount (5%)",                   f"- INR {sum(i['discount_value'] for i in so['items']):,.0f}"],
    ["Net Revenue",                     f"INR {bill['total_net_value']:,.0f}"],
    ["GST Output Tax (18%)",            f"INR {bill['total_tax']:,.0f}"],
    ["Total Invoice Value",             f"INR {bill['total_gross']:,.0f}"],
    ["Cost of Goods Sold (COGS)",       f"INR {sum(m['cogs'] for m in documents[mat_doc_no]['movements']):,.0f}"],
    ["Gross Profit",                    f"INR {bill['total_net_value'] - sum(m['cogs'] for m in documents[mat_doc_no]['movements']):,.0f}"],
    ["Amount Received",                 f"INR {bill['amount_paid']:,.0f}"],
    ["Outstanding A/R",                 f"INR {customers[bill['customer']]['open_ar']:,.0f}"],
]
print()
print(tabulate(summary, headers=["Description", "Amount"], tablefmt="fancy_grid"))


══════════════════════════════════════════════════════════════
  SECTION 10: FINANCIAL SUMMARY
══════════════════════════════════════════════════════════════

╒═════════════════════════════════╤══════════════╕
│ Description                     │ Amount       │
╞═════════════════════════════════╪══════════════╡
│ Gross Revenue (before discount) │ INR 750,000  │
├─────────────────────────────────┼──────────────┤
│ Discount (5%)                   │ - INR 37,500 │
├─────────────────────────────────┼──────────────┤
│ Net Revenue                     │ INR 712,500  │
├─────────────────────────────────┼──────────────┤
│ GST Output Tax (18%)            │ INR 128,250  │
├─────────────────────────────────┼──────────────┤
│ Total Invoice Value             │ INR 840,750  │
├─────────────────────────────────┼──────────────┤
│ Cost of Goods Sold (COGS)       │ INR 750,000  │
├─────────────────────────────────┼──────────────┤
│ Gross Profit                    │ INR -37,500  │
├───────────────────────

In [13]:
# ============================================================
# SECTION 11: INVENTORY STATUS
# ============================================================
print("\n" + "═" * 62)
print("  SECTION 11: INVENTORY STATUS")
print("═" * 62)

inv_rows = [[mat_id, mat["description"], mat["stock"], mat["uom"],
             f"INR {mat['base_price']:,.0f}"]
            for mat_id, mat in materials.items()]
print()
print(tabulate(inv_rows,
               headers=["Material", "Description", "Stock", "UOM", "Unit Price"],
               tablefmt="fancy_grid"))

print("\n" + "=" * 62)
print("  O2C SIMULATION COMPLETE — ALL STEPS EXECUTED SUCCESSFULLY")
print("  KIIT SAP Project | SAP SD — Order-to-Cash Cycle")
print("=" * 62)


══════════════════════════════════════════════════════════════
  SECTION 11: INVENTORY STATUS
══════════════════════════════════════════════════════════════

╒════════════╤════════════════╤═════════╤═══════╤══════════════╕
│ Material   │ Description    │   Stock │ UOM   │ Unit Price   │
╞════════════╪════════════════╪═════════╪═══════╪══════════════╡
│ LAPT-1001  │ Laptop Pro X15 │      40 │ EA    │ INR 75,000   │
╘════════════╧════════════════╧═════════╧═══════╧══════════════╛

  O2C SIMULATION COMPLETE — ALL STEPS EXECUTED SUCCESSFULLY
  KIIT SAP Project | SAP SD — Order-to-Cash Cycle
